In [2]:
!pip install prophet statsmodels

   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   --- ------------------------------------ 1.0/12.1 MB 5.3 MB/s eta 0:00:03
   ----- ---------------------------------- 1.6/12.1 MB 3.7 MB/s eta 0:00:03
   ------ --------------------------------- 2.1/12.1 MB 3.8 MB/s eta 0:00:03
   -------- ------------------------------- 2.6/12.1 MB 3.2 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.1 MB 3.5 MB/s eta 0:00:03
   ---------- ----------------------------- 3.1/12.1 MB 3.5 MB/s eta 0:00:03
   ----------- ---------------------------- 3.4/12.1 MB 2.3 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.1 MB 2.6 MB/s eta 0:00:04
   ------------- -------------------------- 4.2/12.1 MB 2.6 MB/s eta 0:00:04
   ------------------ --------------------- 5.5/12.1 MB 2.6 MB/s eta 0:00:03
   -------------------- ------------------- 6.3/12.1 MB 2.8 MB/s eta 0:00:03
   ----------------------- ---------------- 7.1/12.1 MB 2.8 MB/s eta 0:00:02
   ---

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from prophet import Prophet
from statsmodels.tsa.arima.model import ARIMA
import itertools

PROJECT_ROOT = r"C:\path\to\your\supply_chain_project"   
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
os.makedirs(OUTPUTS_DIR, exist_ok=True)

DATA_PATH = os.path.join(OUTPUTS_DIR, "daily_demand.csv")
HOLDOUT_DAYS = 60
FUTURE_DAYS = 30

demand_df = pd.read_csv(DATA_PATH, parse_dates=["date"])
stores = demand_df["store"].unique()

def mape(actual, pred):
    actual, pred = np.array(actual), np.array(pred)
    mask = actual != 0
    return np.mean(np.abs((actual[mask] - pred[mask]) / actual[mask])) * 100

def rmse(actual, pred):
    return np.sqrt(np.mean((np.array(actual) - np.array(pred)) ** 2))

def best_arima_order(train_series):
    """Small grid search over (p,d,q) - kept intentionally small for speed."""
    best_aic, best_order = np.inf, (1, 1, 1)
    for p, d, q in itertools.product(range(0, 3), range(0, 2), range(0, 3)):
        try:
            model = ARIMA(train_series, order=(p, d, q)).fit()
            if model.aic < best_aic:
                best_aic, best_order = model.aic, (p, d, q)
        except Exception:
            continue
    return best_order

all_results = []
accuracy_rows = []
future_forecast_rows = []

for store in stores:
    s = demand_df[demand_df["store"] == store].sort_values("date").reset_index(drop=True)
    train = s.iloc[: -HOLDOUT_DAYS]
    test = s.iloc[-HOLDOUT_DAYS:]

    # ---------- Prophet ----------
    prophet_train = train.rename(columns={"date": "ds", "demand_units": "y"})[["ds", "y"]]
    m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
    m.fit(prophet_train)

    future = m.make_future_dataframe(periods=HOLDOUT_DAYS + FUTURE_DAYS)
    fcst = m.predict(future)

    prophet_test_pred = fcst.set_index("ds").loc[test["date"], "yhat"].clip(lower=0).values
    prophet_future = fcst.set_index("ds").iloc[-FUTURE_DAYS:]["yhat"].clip(lower=0)

    # ---------- ARIMA ----------
    order = best_arima_order(train["demand_units"])
    arima_model = ARIMA(train["demand_units"], order=order).fit()
    arima_test_pred = arima_model.forecast(steps=HOLDOUT_DAYS).clip(lower=0).values
    # Refit on train+test to project the future 30 days (standard practice)
    arima_full = ARIMA(s["demand_units"], order=order).fit()
    arima_future = arima_full.forecast(steps=FUTURE_DAYS).clip(lower=0)

    # ---------- Record accuracy ----------
    accuracy_rows.append({
        "store": store, "model": "Prophet",
        "MAPE_%": round(mape(test["demand_units"], prophet_test_pred), 2),
        "RMSE": round(rmse(test["demand_units"], prophet_test_pred), 2),
    })
    accuracy_rows.append({
        "store": store, "model": "ARIMA" + str(order),
        "MAPE_%": round(mape(test["demand_units"], arima_test_pred), 2),
        "RMSE": round(rmse(test["demand_units"], arima_test_pred), 2),
    })

    # ---------- Record actual vs predicted (holdout window) ----------
    for i in range(HOLDOUT_DAYS):
        all_results.append({
            "store": store, "date": test["date"].iloc[i],
            "actual": test["demand_units"].iloc[i],
            "prophet_pred": round(prophet_test_pred[i], 1),
            "arima_pred": round(arima_test_pred[i], 1),
        })

    # ---------- Record 30-day future baseline (best model per store) ----------
    prophet_mape_i = mape(test["demand_units"], prophet_test_pred)
    arima_mape_i = mape(test["demand_units"], arima_test_pred)
    chosen = "Prophet" if prophet_mape_i <= arima_mape_i else "ARIMA"
    chosen_future = prophet_future.values if chosen == "Prophet" else arima_future.values
    future_dates = pd.date_range(s["date"].max() + pd.Timedelta(days=1), periods=FUTURE_DAYS)
    for d, v in zip(future_dates, chosen_future):
        future_forecast_rows.append({
            "store": store, "date": d,
            "baseline_forecast_units": round(max(v, 0), 1),
            "model_used": chosen,
        })

    print(f"{store}: Prophet MAPE={prophet_mape_i:.2f}%  ARIMA{order} MAPE={arima_mape_i:.2f}%  -> using {chosen}")

results_df = pd.DataFrame(all_results)
accuracy_df = pd.DataFrame(accuracy_rows)
future_df = pd.DataFrame(future_forecast_rows)

results_df.to_csv(os.path.join(OUTPUTS_DIR, "forecast_results.csv"), index=False)
accuracy_df.to_csv(os.path.join(OUTPUTS_DIR, "forecast_accuracy.csv"), index=False)
future_df.to_csv(os.path.join(OUTPUTS_DIR, "baseline_forecast_30day.csv"), index=False)

print("\nSaved: forecast_results.csv, forecast_accuracy.csv, baseline_forecast_30day.csv")
print("\nAccuracy summary:")
print(accuracy_df.to_string(index=False))


Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:28:36 - cmdstanpy - INFO - Chain [1] start processing
17:28:36 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:28:40 - cmdstanpy - INFO - Chain [1] start processing


Store_1: Prophet MAPE=4.12%  ARIMA(2, 1, 2) MAPE=7.97%  -> using Prophet


17:28:40 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:28:44 - cmdstanpy - INFO - Chain [1] start processing


Store_2: Prophet MAPE=4.38%  ARIMA(2, 1, 2) MAPE=9.33%  -> using Prophet


17:28:44 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:28:47 - cmdstanpy - INFO - Chain [1] start processing


Store_3: Prophet MAPE=3.92%  ARIMA(2, 1, 2) MAPE=7.40%  -> using Prophet


17:28:47 - cmdstanpy - INFO - Chain [1] done processing
Yearly seasonality is enabled with less than 730 days (approximately 2 years) of history. The model may be under-identified, and the trend/seasonality decomposition can be unstable and dependent on the Prophet/Stan version. Consider disabling yearly seasonality or providing more history.
17:28:51 - cmdstanpy - INFO - Chain [1] start processing


Store_4: Prophet MAPE=5.64%  ARIMA(2, 1, 2) MAPE=8.02%  -> using Prophet


17:28:51 - cmdstanpy - INFO - Chain [1] done processing


Store_5: Prophet MAPE=3.89%  ARIMA(2, 1, 2) MAPE=7.65%  -> using Prophet

Saved: forecast_results.csv, forecast_accuracy.csv, baseline_forecast_30day.csv

Accuracy summary:
  store          model  MAPE_%  RMSE
Store_1        Prophet    4.12  7.78
Store_1 ARIMA(2, 1, 2)    7.97 17.28
Store_2        Prophet    4.38  6.28
Store_2 ARIMA(2, 1, 2)    9.33 13.49
Store_3        Prophet    3.92 10.43
Store_3 ARIMA(2, 1, 2)    7.40 20.12
Store_4        Prophet    5.64  5.86
Store_4 ARIMA(2, 1, 2)    8.02  8.64
Store_5        Prophet    3.89  6.78
Store_5 ARIMA(2, 1, 2)    7.65 14.74
